## Challenge Benin-Insight Isheero x Datacamp : Notebook de filtrage des données

### Chargement des données

In [ ]:
# Bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Chargement des données
df_raw = pd.read_csv('/content/drive/MyDrive/Isheero/Data.csv')

# Affichage des premières lignes
print(f"Nombre de lignes brutes : {len(df_raw)}")
print(df_raw.head(3))

Nombre de lignes brutes : 23859
    SQLDATE  YEAR  MonthYear Actor1Name Actor1Type1Code Actor2Name  \
0  20250101  2025     202501      BENIN             NaN        NaN   
1  20250101  2025     202501      BENIN             GOV        NaN   
2  20250101  2025     202501   DIPLOMAT             GOV        NaN   

  Actor2Type1Code  EventCode  EventBaseCode  EventRootCode  ...  NumSources  \
0             NaN         51             51              5  ...           1   
1             NaN         10             10              1  ...           1   
2             NaN         40             40              4  ...           1   

   NumArticles   AvgTone  ActionGeo_Type  ActionGeo_FullName  \
0            5 -7.547170               1               Benin   
1            5 -8.482871               1               Benin   
2            3 -7.843137               1               Benin   

   ActionGeo_CountryCode  ActionGeo_ADM1Code ActionGeo_Lat ActionGeo_Long  \
0                     BN            

### Description du problème

**Le problème** : Le dataset GDELT filtré sur ActionGeo_CountryCode = 'BN' contient une pollution systématique : des articles traitant de Benin City (ville de l'État d'Edo, Nigeria) sont tagués avec les mêmes codes pays que la République du Bénin. Le filtre GPS est inefficace car 91.2% des événements sont placés au centroïde national (9.5°N / 2.25°E), rendant toute bounding box inutile sur ces lignes.

Aucune des variables du dataset ne permet de filtrer efficacement les variables pour consulter ce problème. Pour y parvenir au mieux, Nous appliquons un pipeline de
décontamination en 3 couches progressives.


In [ ]:
import pandas as pd
from datetime import datetime
import os

# Créer le dossier de sortie
os.makedirs('outputs', exist_ok=True)

# Copie de travail — on ne touche jamais à df_raw
df = df_raw.copy()

# Colonnes de traçabilité
df['status']           = 'pending'
df['rejection_layer']  = ''
df['rejection_reason'] = ''
df['confidence_score'] = 0.0
df['llm_label']        = ''

print(f"Dataset chargé : {len(df):,} lignes | {df['status'].value_counts().to_dict()}")

Dataset chargé : 23,859 lignes | {'pending': 23859}


### 1. Elimination des cas évidents

Certaines URLs contiennent des mots-clés qui trahissent
immédiatement leur origine nigériane (Benin City / État d'Edo). Ce filtre élimine les cas les plus évidents.

In [ ]:
BENIN_CITY_URL_KEYWORDS = [
    'benin-city', 'edo-state', '/edo/', 'obaseki',
    'oba-of-benin', 'nscdc', 'benin-kingdom',
    'monday-okpebholo', 'edo-assembly', 'edo-central',
    'edo-police', 'edo-apc', 'edo-revenue',
    'oba-of-benin', 'edo-house'
]

mask_1a = (
    (df['status'] == 'pending') &
    df['SOURCEURL'].apply(
        lambda x: any(kw in str(x).lower() for kw in BENIN_CITY_URL_KEYWORDS)
    )
)

df.loc[mask_1a, 'status']           = 'rejected'
df.loc[mask_1a, 'rejection_layer']  = '1A'
df.loc[mask_1a, 'rejection_reason'] = 'Mot-clé Benin City détecté dans URL'

# Résultat
n_1a = mask_1a.sum()
print(f" Rejetés : {n_1a:,} lignes ({n_1a/len(df)*100:.1f}%)")
print(f"Restants en attente : {(df['status']=='pending').sum():,} lignes")

 Rejetés : 1,356 lignes (5.7%)
Restants en attente : 22,503 lignes


GDELT extrait les noms des acteurs depuis le texte des articles.
Si un acteur porte un nom clairement associé à Benin City ou à l'État d'Edo
(gouverneur, institution locale, titre royal nigérian), la ligne est rejetée.

In [ ]:
BENIN_CITY_ACTOR_KEYWORDS = [
    'BENIN CITY', 'EDO STATE', 'OBASEKI', 'OKPEBHOLO',
    'OBA EWUARE', 'NSCDC', 'EDO ASSEMBLY', 'EDO GOVERNMENT',
    'UNIVERSITY OF BENIN', 'BENIN ELECTRICITY'
]

def has_benin_city_actor(row):
    actor1 = str(row.get('Actor1Name', '')).upper()
    actor2 = str(row.get('Actor2Name', '')).upper()
    return any(
        kw in actor1 or kw in actor2
        for kw in BENIN_CITY_ACTOR_KEYWORDS
    )

mask_1b = (
    (df['status'] == 'pending') &
    df.apply(has_benin_city_actor, axis=1)
)

df.loc[mask_1b, 'status']           = 'rejected'
df.loc[mask_1b, 'rejection_layer']  = '1B'
df.loc[mask_1b, 'rejection_reason'] = 'Acteur Benin City / Edo détecté'

# Résultat
n_1b = mask_1b.sum()
print(f"Rejetés : {n_1b:,} lignes ({n_1b/len(df)*100:.1f}%)")
print(f"Restants en attente : {(df['status']=='pending').sum():,} lignes")

Rejetés : 809 lignes (3.4%)
Restants en attente : 21,694 lignes


###2. Scoring de confiance



Pour les lignes survivantes, on calcule un score de confiance entre **-2.0** et **+2.0** basé sur des signaux positifs (indicateurs de la République du Bénin) et des signaux négatifs (indicateurs du Nigeria).

| Signal | Poids | Description |
|---|---|---|
| URL en `.bj` | +0.5 | Domaine officiel béninois |
| Code ADM1 béninois | +0.3 | Département identifié |
| Acteur explicitement BEN | +0.3 | Nationalité béninoise confirmée |
| Acteur NGA | -0.3 | Acteur nigérian (signal modéré) |
| Domaine nigérian connu + pas de signal positif | -0.5 | Source nigériane sans ancrage béninois |
| Mot-clé Edo/Benin City dans l'URL résiduelle | -0.4 | Signal résiduel non capté en 1A |

**Seuil de rejet : `confidence_score <= 0.0`**

In [ ]:
BENIN_ADM1_CODES = [
    'BC01','BC02','BC03','BC04','BC05','BC06',
    'BC07','BC08','BC09','BC10','BC11','BC12'
]

NIGERIAN_DOMAINS = [
    'dailypost.ng', 'punchng.com', 'premiumtimesng.com',
    'saharareporters.com', 'vanguardngr.com', 'nigerianobservernews.com',
    'guardian.ng', 'thesun.ng', 'blueprint.ng', 'leadership.ng',
    'thenationonlineng.net', 'peoplesdailyng.com', 'thisdaylive.com',
    'thecable.ng', 'theeagleonline.com.ng', 'tribuneonlineng.com',
    'dailytrust.com', 'nationalaccordnewspaper.com', 'channelstv.com',
    'businessdayng.com', 'prompnewsonline.com'
]

BENINESE_DOMAINS = [
    'ortb.bj', 'fraternite.bj', 'lanouvelletribune.info',
    'beninwebtv.com', '24haubenin.com', 'acotonou.com',
    'matin.bj', 'levenementprecis.com', 'gouv.bj',
    'beninplus.com', 'lequotidien.bj'
]

RESIDUAL_EDO_KEYWORDS = [
    'obaseki', 'okpebholo', 'benin-city',
    'edo-', '-edo-', 'oba-of-benin'
]

def compute_confidence_score(row):
    score  = 0.0
    detail = []
    url    = str(row.get('SOURCEURL', '')).lower()
    a1     = str(row.get('Actor1CountryCode', ''))
    adm1   = str(row.get('ActionGeo_ADM1Code', ''))

    #  Signaux positifs
    if '.bj' in url:
        score += 0.5
        detail.append('+0.5 URL .bj')

    if adm1 in BENIN_ADM1_CODES:
        score += 0.3
        detail.append('+0.3 ADM1 béninois')

    if a1 == 'BEN':
        score += 0.3
        detail.append('+0.3 acteur BEN confirmé')

    if any(d in url for d in BENINESE_DOMAINS):
        score += 0.4
        detail.append('+0.4 domaine béninois connu')

    #  Signaux négatifs
    if a1 == 'NGA':
        score -= 0.3
        detail.append('-0.3 acteur NGA')

    is_nigerian_source = any(d in url for d in NIGERIAN_DOMAINS)
    has_positive_signal = score > 0

    if is_nigerian_source and not has_positive_signal:
        score -= 0.5
        detail.append('-0.5 source nigériane sans signal béninois')

    if any(kw in url for kw in RESIDUAL_EDO_KEYWORDS):
        score -= 0.4
        detail.append('-0.4 mot-clé Edo résiduel dans URL')

    return round(score, 2), ' | '.join(detail)

# Appliquer sur les lignes encore en attente
pending_mask = df['status'] == 'pending'
scores = df[pending_mask].apply(
    lambda row: compute_confidence_score(row), axis=1
)

df.loc[pending_mask, 'confidence_score'] = scores.apply(lambda x: x[0])
df.loc[pending_mask, 'rejection_reason'] = scores.apply(lambda x: x[1])

# Rejeter les lignes avec score <= 0
mask_2_rejected = pending_mask & (df['confidence_score'] <= 0.0)
df.loc[mask_2_rejected, 'status']          = 'rejected'
df.loc[mask_2_rejected, 'rejection_layer'] = '2'

# Résultat
n_2 = mask_2_rejected.sum()
print(f" Rejetés : {n_2:,} lignes ({n_2/len(df)*100:.1f}%)")
print(f"Restants en attente : {(df['status']=='pending').sum():,} lignes")

# Distribution des scores pour les survivants
survivors = df[df['status'] == 'pending']['confidence_score']
print(f"\n Distribution des scores (survivants) :")
print(survivors.describe().round(3))

 Rejetés : 20,421 lignes (85.6%)
Restants en attente : 1,273 lignes

 Distribution des scores (survivants) :
count    1273.000
mean        0.490
std         0.192
min         0.400
25%         0.400
50%         0.400
75%         0.400
max         0.900
Name: confidence_score, dtype: float64


On constate que 85% des éléments de notre dataset ont été rejetés. Ce qui est assez étrange. On exporte donc le fichier csv des éléments rejetés pour voir si le scoring trop strict n'a pas rejeté des évènement béninois.

In [ ]:
# Export rapide des rejetés Couche 2 pour vérification manuelle

df_rejected_c2 = df[df['rejection_layer'] == '2'].copy()

print(f"Rejetés Couche 2 : {len(df_rejected_c2):,} lignes")
print(f"\nDistribution des scores :")
print(df_rejected_c2['confidence_score'].value_counts().sort_index())

print(f"\nTop domaines rejetés :")
rejected_domains = df_rejected_c2['SOURCEURL'].str.extract(r'https?://(?:www\.)?([^/]+)')[0]
print(rejected_domains.value_counts().head(20))

# Export
df_rejected_c2.to_csv('outputs/verification_couche2.csv', index=False, encoding='utf-8-sig')
print(f"\n Fichier exporté → outputs/verification_couche2.csv")

Rejetés Couche 2 : 20,421 lignes

Distribution des scores :
confidence_score
-0.9     2888
-0.5     5661
-0.4      663
 0.0    11209
Name: count, dtype: int64

Top domaines rejetés :
0
punchng.com                 1344
dailypost.ng                1213
nigerianobservernews.com     749
allafrica.com                649
leadership.ng                640
guardian.ng                  559
saharareporters.com          485
thisdaylive.com              456
thesun.ng                    435
blueprint.ng                 413
premiumtimesng.com           383
quicknews-africa.net         345
thenationonlineng.net        344
theeagleonline.com.ng        287
thecable.ng                  272
dailytrust.com               271
tribuneonlineng.com          219
promptnewsonline.com         218
fr.allafrica.com             207
legit.ng                     206
Name: count, dtype: int64

 Fichier exporté → outputs/verification_couche2.csv


In [ ]:
# DIAGNOSTIC COUCHE 2 — Comprendre les score = 0.0


df_c2 = df[df['rejection_layer'] == '2'].copy()

# Isoler les score = 0.0 (ni signal positif ni négatif)
df_score_zero = df_c2[df_c2['confidence_score'] == 0.0]
print(f"Lignes score = 0.0 : {len(df_score_zero):,}")

print(f"\nTop domaines score = 0.0 :")
domains_zero = df_score_zero['SOURCEURL'].str.extract(r'https?://(?:www\.)?([^/]+)')[0]
print(domains_zero.value_counts().head(20))

print(f"\nExemples d'URLs score = 0.0 :")
for url in df_score_zero['SOURCEURL'].sample(20, random_state=42):
    print(f"  {url}")

# Isoler les score = -0.5 (uniquement source nigériane sans signal positif)
df_score_half = df_c2[df_c2['confidence_score'] == -0.5]
print(f"\nLignes score = -0.5 : {len(df_score_half):,}")
print(f"\nExemples d'URLs score = -0.5 :")
for url in df_score_half['SOURCEURL'].sample(min(20, len(df_score_half)), random_state=42):
    print(f"  {url}")

Lignes score = 0.0 : 11,209

Top domaines score = 0.0 :
0
allafrica.com              649
quicknews-africa.net       228
fr.allafrica.com           207
legit.ng                   193
yahoo.com                  191
newsghana.com.gh           154
promptnewsonline.com       150
afrik.com                  137
opinionnigeria.com         136
africatopsuccess.com       123
ecofinagency.com           121
myjoyonline.com            116
dw.com                     108
24haubenin.info            102
en.antaranews.com          102
tanzanianewsreports.com     99
ghanaweb.com                96
africa-newsroom.com         94
nigerianeye.com             87
afriquinfos.com             86
Name: count, dtype: int64

Exemples d'URLs score = 0.0 :
  https://www.jeuneafrique.com/1688163/culture/133-ans-apres-le-katakle-du-tresor-royal-dabomey-revient-au-benin/
  http://thepeninsulaqatar.com/article/19/05/2025/amir-president-of-republic-of-benin-discuss-bilateral-cooperation
  https://www.afriquesenlutte.org/a

Après vérification manuelle, le verdict est sans appel : le scoring est bien trop strict. Il a rejeté tous les évènement ambigus. Nous mettons donc en place un nouveau scoring beaucoup plus large que le précédent.

In [ ]:
# Nouvelle couche 2 — Amélioration du scoring grâce aux résultats précédents
#Principe : on ne rejette que ce qu'on prouve être Nigeria

BENIN_ADM1_CODES = [
    'BC01','BC02','BC03','BC04','BC05','BC06',
    'BC07','BC08','BC09','BC10','BC11','BC12'
]

BENINESE_DOMAINS = [
    'ortb.bj', 'fraternite.bj', 'lanouvelletribune.info',
    'beninwebtv.com', '24haubenin.com', 'acotonou.com',
    'matin.bj', 'levenementprecis.com', 'gouv.bj',
    'beninplus.com', 'lequotidien.bj', '24haubenin.info',
    'agenceafrique.com', 'africatopsuccess.com'
]

# Domaines nigérians dont on sait qu'ils couvrent
# AUSSI la République du Bénin — on ne les pénalise pas
NIGERIAN_DOMAINS_AMBIGUOUS = [
    'dailypost.ng', 'punchng.com', 'premiumtimesng.com',
    'saharareporters.com', 'guardian.ng', 'blueprint.ng',
    'thecable.ng', 'leadership.ng', 'thisdaylive.com',
    'thenationonlineng.net', 'thesun.ng', 'tribuneonlineng.com',
    'dailytrust.com', 'vanguardngr.com'
]

# Domaines nigérians qui couvrent EXCLUSIVEMENT
# Benin City / État d'Edo — ceux-là on les pénalise
NIGERIAN_DOMAINS_EDO_ONLY = [
    'nigerianobservernews.com',   # Basé à Benin City
    'peoplesdailyng.com',
    'theeagleonline.com.ng',
    'promptnewsonline.com',
    'nationalaccordnewspaper.com'
]

# Mots-clés dans l'URL qui prouvent Benin City
# (complément de la Couche 1)
EDO_URL_RESIDUAL = [
    'edo-', '-edo', '/edo', 'obaseki', 'okpebholo',
    'benin-city', 'oba-of-benin', 'nscdc', 'oba-ewuare'
]

# Mots-clés dans l'URL qui prouvent République du Bénin
BENIN_REPUBLIC_URL_SIGNALS = [
    'cotonou', 'porto-novo', 'patrice-talon', 'talon',
    'benin-republic', 'republic-of-benin', 'beninese',
    'benin-coup', 'jihadist', 'tri-border', 'w-park',
    'abomey', 'parakou', 'natitingou',
]

def compute_confidence_score_v2(row):
    score  = 0.0
    detail = []
    url    = str(row.get('SOURCEURL', '')).lower()
    a1     = str(row.get('Actor1CountryCode', ''))
    a2     = str(row.get('Actor2CountryCode', ''))
    adm1   = str(row.get('ActionGeo_ADM1Code', ''))

    #  SIGNAUX FORTEMENT POSITIFS

    # Domaine béninois connu
    if any(d in url for d in BENINESE_DOMAINS):
        score += 0.8
        detail.append('+0.8 domaine béninois')

    # URL en .bj
    if '.bj/' in url or url.endswith('.bj'):
        score += 0.6
        detail.append('+0.6 URL .bj')

    # Département béninois dans ADM1
    if adm1 in BENIN_ADM1_CODES:
        score += 0.5
        detail.append('+0.5 ADM1 béninois')

    # Acteur explicitement béninois
    if a1 == 'BEN' or a2 == 'BEN':
        score += 0.4
        detail.append('+0.4 acteur BEN')

    # Mot-clé République du Bénin dans l'URL
    if any(kw in url for kw in BENIN_REPUBLIC_URL_SIGNALS):
        score += 0.5
        detail.append('+0.5 mot-clé Bénin République dans URL')

    # SIGNAUX NÉGATIFS (pollution prouvée uniquement)

    # Domaine Edo-only (couvre exclusivement Benin City)
    if any(d in url for d in NIGERIAN_DOMAINS_EDO_ONLY):
        score -= 0.6
        detail.append('-0.6 domaine Edo-only')

    # Mot-clé Edo résiduel dans URL (non capté en 1A)
    if any(kw in url for kw in EDO_URL_RESIDUAL):
        score -= 0.7
        detail.append('-0.7 mot-clé Edo résiduel URL')

    # Acteur NGA SANS aucun signal positif béninois
    if (a1 == 'NGA' or a2 == 'NGA') and score == 0.0:
        score -= 0.2
        detail.append('-0.2 acteur NGA sans signal béninois')

    # RÈGLE CRITIQUE : score 0.0 = on conserve
    # Une ligne sans signal dans aucun sens est ambiguë,
    # pas prouvée nigériane. On la garde pour la Couche 3.

    return round(score, 2), ' | '.join(detail) if detail else 'Aucun signal'

#  Réinitialiser les rejetés Couche 2 uniquement
mask_was_c2 = df['rejection_layer'] == '2'
df.loc[mask_was_c2, 'status']           = 'pending'
df.loc[mask_was_c2, 'rejection_layer']  = ''
df.loc[mask_was_c2, 'rejection_reason'] = ''
df.loc[mask_was_c2, 'confidence_score'] = 0.0
print(f" Réinitialisés pour re-scoring : {mask_was_c2.sum():,} lignes")

#  Appliquer le nouveau scoring
pending_mask = df['status'] == 'pending'
results = df[pending_mask].apply(
    lambda row: compute_confidence_score_v2(row), axis=1
)
df.loc[pending_mask, 'confidence_score'] = results.apply(lambda x: x[0])
df.loc[pending_mask, 'rejection_reason'] = results.apply(lambda x: x[1])

# Nouveau seuil : on rejette seulement si score < 0
# Les score = 0.0 (ambigus) passent en Couche 3
mask_2_rejected = pending_mask & (df['confidence_score'] < 0.0)
df.loc[mask_2_rejected, 'status']          = 'rejected'
df.loc[mask_2_rejected, 'rejection_layer'] = '2'

n_2 = mask_2_rejected.sum()
n_ambiguous = (pending_mask & (df['confidence_score'] == 0.0)).sum()
n_positive  = (pending_mask & (df['confidence_score'] > 0.0)).sum()

print(f"\n Résultats Couche 2 corrigée :")
print(f"   Rejetés (score < 0)   : {n_2:,} lignes")
print(f"   Ambigus (score = 0.0) : {n_ambiguous:,} (partent en couche 3)")
print(f"   Positifs (score > 0)  : {n_positive:,} (conservés directement)")
print(f"   Total encore pending  : {pending_mask.sum() - n_2:,} lignes")

#  Distribution pour vérification
print(f"\n Distribution des scores :")
print(df[pending_mask]['confidence_score'].value_counts().sort_index())

 Réinitialisés pour re-scoring : 20,421 lignes

 Résultats Couche 2 corrigée :
   Rejetés (score < 0)   : 5,090 lignes
   Ambigus (score = 0.0) : 13,570 (partent en couche 3)
   Positifs (score > 0)  : 3,034 (conservés directement)
   Total encore pending  : 16,604 lignes

 Distribution des scores :
confidence_score
-1.3      935
-0.7     3536
-0.6      569
-0.1       50
 0.0    13570
 0.1        5
 0.5     1495
 0.8     1227
 1.3      156
 1.4      129
 1.9       22
Name: count, dtype: int64


On analyse encore les URLs ambigues à nouveau avec de nouveaux mots clés

In [ ]:
# Analyse des URLs ambiguës pour trouver des signaux supplémentaires
ambiguous_mask = (df['status'] == 'pending') & (df['confidence_score'] == 0.0)
df_ambiguous = df[ambiguous_mask].copy()

# Mots-clés positifs République du Bénin dans le PATH de l'URL
BENIN_REPUBLIC_PATH = [
    'benin-republic', 'republic-of-benin', 'beninese', 'republique-du-benin',
    'cotonou', 'porto-novo', 'patrice-talon', 'talon-benin',
    'abomey', 'parakou', 'natitingou', 'kandi', 'djougou',
    'benin-coup', 'coup-benin', 'jihadist-benin', 'benin-attack',
    'tri-border', 'w-national-park', 'benin-army', 'armee-benin',
    'benin-election', 'election-benin', 'benin-government',
    'gouvernement-benin', 'benin-economy', 'economie-benin',
    'benin-minister', 'ministre-benin', 'benin-president',
    'president-benin', 'benin-niger', 'benin-togo', 'benin-burkina',
    'west-africa-benin', 'afrique-ouest-benin', 'greve-benin',
    'terrorisme-benin', 'securite-benin', 'diplomatie-benin'
]

# Mots-clés négatifs Benin City dans le PATH
BENIN_CITY_PATH = [
    'edo', 'oba-', 'palace', 'benin-kingdom', 'benin-traditional',
    'ewuare', 'oba-palace', 'benin-bronze', 'benin-monarch',
    'ancient-benin', 'benin-artifacts', 'benin-artworks',
    'royal-benin', 'benin-massacre', 'benin-palace',
    'obaseki', 'okpebholo', 'benin-city', 'nscdc-benin'
]

# Domaines panafricains connus pour couvrir la République du Bénin
TRUSTED_PAN_AFRICAN = [
    'allafrica.com', 'fr.allafrica.com', 'jeuneafrique.com',
    'rfi.fr', 'bbc.com', 'dw.com', 'africanews.com',
    'reuters.com', 'apnews.com', 'afp.com', 'voaafrique.com',
    'tv5monde.com', 'france24.com', 'lemonde.fr', 'lefigaro.fr',
    'afrik.com', 'afriquinfos.com', 'africatopsuccess.com',
    'ecofinagency.com', 'agenceafrique.com', 'pressafrik.com',
    'leral.net', 'rewmi.com', 'senenews.com', 'myjoyonline.com',
    'ghanaweb.com', 'newsghana.com.gh', 'arise.tv',
    'thepeninsulaqatar.com', 'astanatimes.com', 'devdiscourse.com',
    'quicknews-africa.net', 'africa-newsroom.com'
]

def pre_filter_ambiguous(row):
    url = str(row.get('SOURCEURL', '')).lower()

    # Signal positif fort : domaine panafricain/international de confiance
    if any(d in url for d in TRUSTED_PAN_AFRICAN):
        return 'clean', '+trusted panafricain/international'

    # Signal positif : mot-clé République du Bénin dans le path
    if any(kw in url for kw in BENIN_REPUBLIC_PATH):
        return 'clean', '+mot-clé Bénin République dans URL'

    # Signal négatif : mot-clé Benin City dans le path
    if any(kw in url for kw in BENIN_CITY_PATH):
        return 'rejected', '-mot-clé Benin City dans URL path'

    # Vraiment ambigu : on envoie au LLM
    return 'llm', 'Aucun signal supplémentaire'

results = df_ambiguous.apply(pre_filter_ambiguous, axis=1)
df_ambiguous['pre_filter_status'] = results.apply(lambda x: x[0])
df_ambiguous['pre_filter_reason'] = results.apply(lambda x: x[1])

# Résultats
counts = df_ambiguous['pre_filter_status'].value_counts()
print(f"Résultats pré-filtre sur les ambigus :")
print(f"Clean (conservés)  : {counts.get('clean', 0):,} lignes")
print(f"Rejetés            : {counts.get('rejected', 0):,} lignes")
print(f"LLM nécessaire     : {counts.get('llm', 0):,} lignes")

llm_needed = df_ambiguous[df_ambiguous['pre_filter_status'] == 'llm']
unique_llm_urls = llm_needed['SOURCEURL'].nunique()
temps = (unique_llm_urls / 15) / 60
print(f"\n URLs uniques restant pour LLM : {unique_llm_urls:,}")

Résultats pré-filtre sur les ambigus :
Clean (conservés)  : 2,407 lignes
Rejetés            : 343 lignes
LLM nécessaire     : 10,820 lignes

 URLs uniques restant pour LLM : 4,060


### 3. Web Scrapping

Nous avons là un total de 10820 URLs (4060 uniques)  qui restent ambigues. Pour les départager, nous adoptons une stratégie de web scrapping. La stratégie initiale consistait à envoyer les URLs à un LLM pour qu'il départage, mais cette stratégie à été jugée trop chronophage.
Nous envoyons d'abord une requête au site web pour récupérer le contenu de l'article (code source), puis nous récupérons uniquement le titre et les 4 premiers paragraphes grâce à la bibliothèque beautifulsoup.

Puis, on compare la fréquence de deux dictionnaires de mots-clés :
    *   **République du Bénin** (ex: *Cotonou, Patrice Talon, UEMOA*)
    *   **Benin City** (ex: *Edo State, Oba, Bronzes*)

Si le score "City" domine, l'article est rejeté.
Si le score "Republic" domine ou est neutre, l'article est conservé.

La parallélisation avec 10 threads permet d'aller plus vite.


In [ ]:
import requests
from bs4 import BeautifulSoup
import time
from concurrent.futures import ThreadPoolExecutor, as_completed


BENIN_REPUBLIC_CONTENT = [
    # Villes béninoises
    'cotonou', 'porto-novo', 'parakou', 'abomey', 'natitingou',
    'kandi', 'djougou', 'bohicon', 'lokossa', 'ouidah',
    # Personnalités béninoises
    'patrice talon', 'talon', 'alassane seidou', 'cakpo',
    'boni yayi', 'yayi',
    # Institutions béninoises
    'bénin', 'benin republic', 'republic of benin',
    'republique du benin', 'beninois', 'beninese',
    'gouvernement du bénin', 'armée béninoise',
    'forces armées du bénin', 'criet',
    # Contexte géopolitique béninois
    'tri-border', 'zone des trois frontières',
    'w national park', 'parc w', 'pendjari',
    'jihadiste', 'jihadist', 'terroriste au bénin',
    'coup au bénin', 'tentative de coup',
    'cedeao bénin', 'ecowas benin',
    # Économie béninoise
    'port de cotonou', 'uemoa bénin', 'bceao bénin',
]

BENIN_CITY_CONTENT = [
    # Lieux nigérians
    'benin city', 'edo state', 'edo kingdom',
    'ancient benin', 'benin kingdom',
    # Personnalités nigérianes
    'obaseki', 'okpebholo', 'oba ewuare', 'oba of benin',
    'oshiomhole',
    # Institutions nigérianes
    'edo government', 'edo assembly', 'edo police',
    'university of benin', 'uniben',
    'benin electricity distribution',
    'edo state university', 'nscdc benin',
    # Contexte nigérian
    'benin bronze', 'benin artifacts', 'benin artworks',
    'royal palace benin', 'oba palace',
    'benin massacre', 'punitive expedition',
]

def scrape_and_classify(url, timeout=6):
    """
    Scrape le titre + 2 premiers paragraphes et classe l'article.
    Retourne : 'clean', 'rejected', ou 'unknown'
    """
    try:
        resp = requests.get(
            url, timeout=timeout,
            headers={
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                              'AppleWebKit/537.36 (KHTML, like Gecko) '
                              'Chrome/91.0.4472.124 Safari/537.36'
            }
        )
        soup = BeautifulSoup(resp.text, 'html.parser')

        # Extraire titre + premiers paragraphes
        title = soup.find('title')
        title_text = title.text.strip().lower() if title else ''

        paragraphs = soup.find_all('p')
        content_text = ' '.join([p.text for p in paragraphs[:4]]).lower()

        full_text = f"{title_text} {content_text}"

        # Score de contenu
        benin_rep_hits = [kw for kw in BENIN_REPUBLIC_CONTENT if kw in full_text]
        benin_city_hits = [kw for kw in BENIN_CITY_CONTENT if kw in full_text]

        score_rep  = len(benin_rep_hits)
        score_city = len(benin_city_hits)

        if score_rep == 0 and score_city == 0:
            # Aucun signal dans le contenu donc conserver par défaut
            # (mieux vaut un faux positif qu'un vrai article perdu)
            return 'unknown', 'Aucun signal dans le contenu'

        if score_city > score_rep:
            return 'rejected', f'Benin City ({score_city} hits: {benin_city_hits[:3]})'

        return 'clean', f'Bénin République ({score_rep} hits: {benin_rep_hits[:3]})'

    except Exception as e:
        return 'unknown', f'Erreur scraping: {str(e)[:60]}'



# APPLICATION SUR LES URLs LLM NÉCESSAIRES DE l'ETAPE PRECEDENTE

llm_needed_mask = (
    (df['status'] == 'pending') &
    (df['confidence_score'] == 0.0) &
    ~df['SOURCEURL'].isin(
        df_ambiguous[df_ambiguous['pre_filter_status'] == 'clean']['SOURCEURL']
    ) &
    ~df['SOURCEURL'].isin(
        df_ambiguous[df_ambiguous['pre_filter_status'] == 'rejected']['SOURCEURL']
    )
)

urls_to_scrape = df[llm_needed_mask]['SOURCEURL'].unique()
print(f" URLs à scraper : {len(urls_to_scrape):,}")

# Scraping parallèle (10 threads) pour aller plus vite
url_results = {}
errors = 0

with ThreadPoolExecutor(max_workers=10) as executor:
    future_to_url = {
        executor.submit(scrape_and_classify, url): url
        for url in urls_to_scrape
    }

    for i, future in enumerate(as_completed(future_to_url)):
        url = future_to_url[future]
        try:
            status, reason = future.result()
            url_results[url] = (status, reason)
        except Exception as e:
            url_results[url] = ('unknown', str(e))
            errors += 1

        if i % 200 == 0 and i > 0:
            classified = {v[0]: 0 for v in [('clean',), ('rejected',), ('unknown',)]}
            for v in url_results.values():
                classified[v[0]] = classified.get(v[0], 0) + 1
            print(f"  Progression {i}/{len(urls_to_scrape)} | "
                  f" Clean{classified['clean']} | "
                  f" Rejetés {classified['rejected']} | "
                  f"Conservés {classified['unknown']}")

print(f"\n Scraping terminé — Erreurs : {errors}")

# Résumé
from collections import Counter
counts = Counter(v[0] for v in url_results.values())
print(f"\n Résultats scraping :")
print(f"    Clean    : {counts['clean']:,} URLs")
print(f"    Rejetés  : {counts['rejected']:,} URLs")
print(f"    Inconnus : {counts['unknown']:,} URLs (conservés par défaut)")


# INTÉGRATION DES RÉSULTATS DANS LE DATAFRAME

# Appliquer les résultats du scraping
for url, (status, reason) in url_results.items():
    mask = llm_needed_mask & (df['SOURCEURL'] == url)
    if status == 'rejected':
        df.loc[mask, 'status']           = 'rejected'
        df.loc[mask, 'rejection_layer']  = '3'
        df.loc[mask, 'rejection_reason'] = f'Scraping contenu: {reason}'

# Appliquer les résultats du pré-filtre (clean et rejected)
for _, row in df_ambiguous.iterrows():
    url = row['SOURCEURL']
    pre_status = row['pre_filter_status']
    pre_reason = row['pre_filter_reason']

    mask = (df['SOURCEURL'] == url) & (df['status'] == 'pending')
    if pre_status == 'rejected':
        df.loc[mask, 'status']           = 'rejected'
        df.loc[mask, 'rejection_layer']  = '3'
        df.loc[mask, 'rejection_reason'] = f'Pré-filtre: {pre_reason}'
    # Les 'clean' et 'unknown' restent pending → seront marqués clean à la fin

# BILAN FINAL


df.loc[df['status'] == 'pending', 'status'] = 'clean'

total = len(df)
n_clean    = (df['status'] == 'clean').sum()
n_rejected = (df['status'] == 'rejected').sum()


print(f" BILAN FINAL DU PIPELINE")
print(f"Dataset brut         : {total:,} lignes")
print(f"Dataset clean     : {n_clean:,} lignes ({n_clean/total*100:.1f}%)")
print(f"Événements rejetés : {n_rejected:,} lignes ({n_rejected/total*100:.1f}%)")
print(f"\nDétail par couche :")
for layer in ['1A', '1B', '2', '3']:
    n = (df['rejection_layer'] == layer).sum()
    print(f"  Couche {layer} : {n:,} lignes rejetées")

 URLs à scraper : 4,060
  Progression 200/4060 |  Clean52 |  Rejetés 8 | Conservés 141
  Progression 400/4060 |  Clean86 |  Rejetés 22 | Conservés 293
  Progression 600/4060 |  Clean113 |  Rejetés 49 | Conservés 439
  Progression 800/4060 |  Clean150 |  Rejetés 84 | Conservés 567
  Progression 1000/4060 |  Clean167 |  Rejetés 98 | Conservés 736
  Progression 1200/4060 |  Clean200 |  Rejetés 118 | Conservés 883
  Progression 1400/4060 |  Clean239 |  Rejetés 141 | Conservés 1021
  Progression 1600/4060 |  Clean276 |  Rejetés 153 | Conservés 1172
  Progression 1800/4060 |  Clean306 |  Rejetés 165 | Conservés 1330
  Progression 2000/4060 |  Clean345 |  Rejetés 185 | Conservés 1471
  Progression 2200/4060 |  Clean377 |  Rejetés 205 | Conservés 1619
  Progression 2400/4060 |  Clean413 |  Rejetés 227 | Conservés 1761
  Progression 2600/4060 |  Clean440 |  Rejetés 260 | Conservés 1901
  Progression 2800/4060 |  Clean456 |  Rejetés 274 | Conservés 2071
  Progression 3000/4060 |  Clean488 |  Rej

### Finalisation et export

In [ ]:


import pandas as pd
from datetime import datetime
import os

os.makedirs('outputs', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# Marquer tous les pending restants comme clean
df.loc[df['status'] == 'pending', 'status'] = 'clean'

# Fichier 1 : Dataset final purifié
df_clean = df[df['status'] == 'clean'].drop(
    columns=['status', 'rejection_layer',
             'rejection_reason', 'confidence_score', 'llm_label'],
    errors='ignore'
)
clean_path = f'outputs/beninwatch_clean_{timestamp}.csv'
df_clean.to_csv(clean_path, index=False, encoding='utf-8-sig')
print(f" Dataset clean     : {len(df_clean):,} lignes → {clean_path}")

# Fichier 2 : Événements rejetés avec traçabilité
df_rejected = df[df['status'] == 'rejected']
rejected_path = f'outputs/beninwatch_rejected_{timestamp}.csv'
df_rejected.to_csv(rejected_path, index=False, encoding='utf-8-sig')
print(f" Événements rejetés : {len(df_rejected):,} lignes → {rejected_path}")

#Fichier 3 : Log de synthèse
total = len(df)
log_rows = []
descriptions = {
    '1A': 'Mots-clés Benin City dans URL',
    '1B': 'Acteurs Benin City / Edo détectés',
    '2':  'Score de confiance négatif',
    '3':  'Pré-filtre + Scraping contenu'
}
for layer, desc in descriptions.items():
    n = (df['rejection_layer'] == layer).sum()
    log_rows.append({
        'Couche':        layer,
        'Description':   desc,
        'Rejetés':       n,
        'Part du total': f"{n/total*100:.1f}%"
    })
log_rows.append({
    'Couche':        'TOTAL REJETÉS',
    'Description':   'Tous rejetés confondus',
    'Rejetés':       len(df_rejected),
    'Part du total': f"{len(df_rejected)/total*100:.1f}%"
})
log_rows.append({
    'Couche':        'CLEAN',
    'Description':   'Événements conservés',
    'Rejetés':       len(df_clean),
    'Part du total': f"{len(df_clean)/total*100:.1f}%"
})

df_log = pd.DataFrame(log_rows)
log_path = f'outputs/beninwatch_rejection_log_{timestamp}.csv'
df_log.to_csv(log_path, index=False, encoding='utf-8-sig')

print(f"\nLog de synthèse :")
print(df_log.to_string(index=False))
print(f"\nTaux de conservation    : {len(df_clean)/total*100:.1f}%")
print(f"Taux de décontamination : {len(df_rejected)/total*100:.1f}%")

 Dataset clean     : 15,188 lignes → outputs/beninwatch_clean_20260502_0832.csv
 Événements rejetés : 8,671 lignes → outputs/beninwatch_rejected_20260502_0832.csv

Log de synthèse :
       Couche                       Description  Rejetés Part du total
           1A     Mots-clés Benin City dans URL     1356          5.7%
           1B Acteurs Benin City / Edo détectés      809          3.4%
            2        Score de confiance négatif     5090         21.3%
            3     Pré-filtre + Scraping contenu     1416          5.9%
TOTAL REJETÉS            Tous rejetés confondus     8671         36.3%
        CLEAN              Événements conservés    15188         63.7%

Taux de conservation    : 63.7%
Taux de décontamination : 36.3%


## Conclusion

Les 3 fichiers csv (clean, rejected et log) ont été exportés efficacement.

Une analyse manuelle a ensuite confirmé que le filtrage a été bien réalisé.